# Exploratory Graph: Warming by Latitude and Month

**Question:** How does climate warming vary across seasons — and does the global average hide where and when it happens?

- **Data:** CMIP6 GFDL-ESM4, `historical` experiment, member `r1i1p1f1`, monthly near-surface air temperature (`tas`)
- **Source:** NOAA public CMIP6 bucket on Google Cloud (`gs://cmip6`), read directly with no download
- **Transformation:** zonal (east–west) mean at each latitude → average 1980–2014 minus average 1850–1900, for each calendar month
- **Output:** `lat_month_warming.png`, `lat_month_warming.csv`

In [1]:
%pip install -q xarray gcsfs zarr cftime pandas matplotlib
import xarray as xr, gcsfs, numpy as np, pandas as pd, matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'xarray'

## 1. Load data

In [ ]:
fs = gcsfs.GCSFileSystem(token='anon')   # anonymous access to the public bucket
path = 'gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/historical/r1i1p1f1/Amon/tas/gr1/v20190726/'
ds = xr.open_zarr(fs.get_mapper(path), consolidated=True)
ds

## 2. Compute warming by latitude and month

In [ ]:
# Zonal mean in Celsius -> array of shape (months, lat)
zonal = (ds.tas - 273.15).mean('lon').compute().values
lat = ds.lat.values

# Monthly data run Jan 1850 - Dec 2014, so reshape to (year, month, lat)
M = zonal.reshape(-1, 12, len(lat))
years = np.arange(1850, 1850 + M.shape[0])

baseline = M[(years >= 1850) & (years <= 1900)].mean(axis=0)
recent   = M[(years >= 1980) & (years <= 2014)].mean(axis=0)
dT = recent - baseline          # shape (12 months, lat)

## 3. Plot

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
vmax = np.abs(dT).max()   # symmetric color scale so 0 = white

fig, ax = plt.subplots(figsize=(10, 5))
pc = ax.pcolormesh(np.arange(12), lat, dT.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='nearest')
ax.set_xticks(range(12)); ax.set_xticklabels(months)
ax.set_yticks([-90, -60, -30, 0, 30, 60, 90])
ax.set_yticklabels(['90S', '60S', '30S', 'EQ', '30N', '60N', '90N'])
ax.set_ylabel('Latitude')
ax.set_title('Where and When Has It Warmed Most? Warming by Latitude and Month (1980-2014)')
plt.colorbar(pc, label='Temperature anomaly (°C)')
fig.text(0.5, -0.01, 'CMIP6 GFDL-ESM4 | Baseline: 1850-1900 | Zonal (east-west) average', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('lat_month_warming.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Export data

In [ ]:
df = pd.DataFrame(dT.T, columns=months)
df.insert(0, 'lat', lat)
df.round(3).to_csv('lat_month_warming.csv', index=False)
print(df.round(3).to_csv(index=False))

## Observations

- **Both poles warm most in their own cold season.** The Arctic (60–75°N) warms most in Nov–Feb; the Antarctic and Southern Ocean (60–90°S) warm most in Apr–Sep (Southern Hemisphere winter).
- **The tropics warm evenly (about +0.5°C) all year**, with little seasonal difference.
- **The global average hides this.** Seasons are reversed between hemispheres, so a global "winter" (Dec–Feb) mixes Northern winter with Southern summer.
- **Caveat:** this is one model run. Some features, such as slight spring cooling at northern mid-latitudes, may be natural variability.